In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-08-13 19:44:37 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Actualización

In [2]:
### LEER ATENTAMENTE ###
# Las tablas que crea este jupyter son: `proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist` y `proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist`
# Modificar los valores de las siguientes variables cada vez que se creen históricos o se realicen actualizaciones.

# ¿Va a actualizar o construir histórico?
actualizar = False # MODIFICAR. Si se desea crear el histórico, poner False. Si se van a actualizar, poner True.

# Almacenar histórico de métricas de uso de wompi
fecha_inicial_metricas = '2022-01-01' # MODIFICAR. INICIO DE UN MES. EN ACTUALIZACIÓN USAR EL SIGUIENTE MES DESPUÉS DEL ÚLTIMO ALMACENADO
fecha_final_metricas = '2026-07-31' # MODIFICAR. FIN DE UN MES. EN ACTUALIZACIÓN USAR EL MES RECIENTE CON TRANSACCIONES COMPLETAS

# El ciclo de vida de la zona proceso_vdm es de 30 días
# La última vez que se borró y creó o insertó información en las tablas 
# fue el 2026-08-13 # MODIFICAR.


# Hallazgos

Tabla `resultados_wompi.wompi_transactions`

- La tabla se ingesta incrementalmente. 
- Se debe adicionar el filtro `estado_transaccion = 'Aprobada'`
- El monto_transaccion está en centavos, se debe dividir en 100 para transformarlo en pesos
- La variable que representa la fecha de transaccion es `fecha_creacion_transaccion`
- El detalle de la fecha de la transacción es hasta minutos
- La información se ingesta t menos 1 día [sucede en octubre y noviembre de 2025, seguramente en los siguientes meses], esto se mantiene desde el 2023. Sin embargo, se observa que para esos mismos meses del año 2022 la ingestión de las compras diarias se realiza el último día de cada mes.
- En el campo `creado` se encuentra la fecha del primer registro del id_comercio en la plataforma, esto no significa que el cliente, en esa fecha, está listo para recibir transacciones. Por esta razón, está fecha corresponderá a la `fecha vinculacion primer registro`


## Análisis ingestión compras tabla transaccional wompi

In [3]:
sql = f"""
WITH outcome1 AS
  (SELECT YEAR,
          MONTH,
          DAY,
          fecha_creacion_transaccion,
          count(*) AS num_compras
   FROM resultados_wompi.wompi_transactions
   WHERE YEAR = 2026
     AND MONTH BETWEEN 7 AND 8
     AND DAY BETWEEN 1 AND 31
   GROUP BY 1,
            2,
            3,
            4
   ORDER BY fecha_creacion_transaccion DESC, YEAR DESC, MONTH DESC, DAY DESC),
     outcome2 AS
  (SELECT *,
          sum(num_compras) OVER (PARTITION BY fecha_creacion_transaccion) AS total_compras,
                                sum(num_compras) OVER (PARTITION BY fecha_creacion_transaccion
                                                       ORDER BY YEAR,
                                                                MONTH,
                                                                DAY ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_compras_cumsum
   FROM outcome1
   ORDER BY fecha_creacion_transaccion DESC, YEAR DESC, MONTH DESC, DAY DESC)
SELECT *,
       row_number() OVER (PARTITION BY fecha_creacion_transaccion
                          ORDER BY YEAR,
                                   MONTH,
                                   DAY) AS num_ing,
                         round(num_compras /total_compras, 4) AS prop,
                         round(num_compras_cumsum / total_compras, 4) AS prop_cumsum
FROM outcome2
ORDER BY fecha_creacion_transaccion DESC,
         YEAR DESC, MONTH DESC, DAY DESC;
"""
df_ingestiones = helper.obtener_dataframe(sql)

2026-08-13 19:45:34 - [INFO] - Transcurrido: 1786668334, Tiempo de Refresco = 1000


------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 1/1 DATAFRAME        descargando   07:45:34 PM             

2026-08-13 19:45:38 - [INFO] - 45 filas, 10 columnas, 00:02.9 consultando, 00:00.6 descargando, 00:00.0 convirtiendo


 1/1 DATAFRAME         finalizado   07:45:34 PM     00:03.8 
------------------------------------------------------------


In [4]:
df_ingestiones

,year,month,day,fecha_creacion_transaccion,num_compras,total_compras,num_compras_cumsum,num_ing,prop,prop_cumsum
0,2026,8,9,NaN,1,1,1,1,1.0,1.0
1,2026,8,12,20260812.0,492716,492716,492716,1,1.0,1.0
2,2026,8,11,20260811.0,530453,530453,530453,1,1.0,1.0
3,2026,8,10,20260810.0,530024,530024,530024,1,1.0,1.0
4,2026,8,9,20260809.0,368015,368015,368015,1,1.0,1.0
5,2026,8,8,20260808.0,473685,473685,473685,1,1.0,1.0
6,2026,8,7,20260807.0,451308,451308,451308,1,1.0,1.0
7,2026,8,6,20260806.0,623083,623083,623083,1,1.0,1.0
8,2026,8,5,20260805.0,1379118,1379118,1379118,1,1.0,1.0
9,2026,8,4,20260804.0,654107,654107,654107,1,1.0,1.0


## Construcción histórico transacciones

In [5]:
#### OMITIR ESTA INFORMACIÓN ####
# # Construcción histórico trxs
# # Se selecciona el rango de tiempo de las ingestiones a almacenar [Esto para efectos de facilitar la actualización del histórico]
# fecha_inicial = '2025-11-15' # MODIFICAR. DEBE SER EL PRIMER DÍA DE INGESTIÓN DE TRANSACCIONES A ALMACENAR O EL SIGUIENTE DÍA DESPUÉS DEL ÚLTIMO EN UNA ACTUALIZACIÓN.
# fecha_final = '2025-11-30' # MODIFICAR. DEBE SER EL ÚLTIMO DÍA DE INGESTIÓN DE TRANSACCIONES ALMACENADAS O EL DÍA MÁS RECIENTE EN UNA ACTUALIZACIÓN
# fecha_inicial_ts = pd.to_datetime(fecha_inicial)
# fecha_final_ts = pd.to_datetime(fecha_final)
# fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='D')

# # # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# # pri_dia_part = fechas[-1] + relativedelta(days=1)
# # pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# # ult_dia_part = fechas[-1] + relativedelta(days=10)
# # ult_dia_part = ult_dia_part.date().isoformat()
# # fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# # fechas = fechas.append(fechas_faltantes)

# # df_config
# df_config = pd.DataFrame({'fechas': fechas})
# df_config['year'] = df_config['fechas'].dt.year
# df_config['month'] = df_config['fechas'].dt.month
# df_config['day'] = df_config['fechas'].dt.day
# # df_config['fechas_fin_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1) - relativedelta(days=1))
# # df_config['year_fin_mes'] = df_config['fechas_fin_mes'].dt.year
# # df_config['month_fin_mes'] = df_config['fechas_fin_mes'].dt.month
# # df_config['day_fin_mes'] = df_config['fechas_fin_mes'].dt.day
# df_config
#### OMITIR ESTA INFORMACIÓN ####

In [5]:
print('#' * 50)
print('')
print('Obteniendo transacciones wompi de los comercios')
print('')

sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_trxs PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_trxs STORED AS PARQUET AS
SELECT cast(id_comercio as varchar) as cod_unico,
    fecha_creacion_transaccion as f_trx,
    count(*) AS num_trxs,
    CAST(sum(monto_transaccion)/100 AS DECIMAL(38,2)) AS mnt_total_trxs
FROM resultados_wompi.wompi_transactions
WHERE LOWER(TRIM(estado_transaccion)) = "aprobada"
AND YEAR >= 2022
AND MONTH BETWEEN 1 AND 12
AND DAY BETWEEN 1 AND 31
GROUP BY 1,
            2;"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_trxs;"""
helper.ejecutar_consulta(sql_compute)


##################################################

Obteniendo transacciones wompi de los comercios

----------------------------------------------------------------------------------------------
  i    tipo                    nombre                     estado     hora_inicio   duracion   
----------------------------------------------------------------------------------------------
 2/2      DROP proceso.mdo_wompi_vinc_primer_regi_trxs   finalizado   07:50:52 PM     00:00.6 
----------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------
  i    tipo                    nombre                     estado     hora_inicio   duracion   
----------------------------------------------------------------------------------------------
 3/3    CREATE proceso.mdo_wompi_vinc_primer_regi_trxs   finalizado   07:50:53 PM     00:05.3 
--------------------------------------------

## Construcción histórico transacciones por mes

In [6]:
print('#' * 50)
print('')
print('Obteniendo transacciones wompi de los comercios')
print('')

sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_trxs_mes_1 PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_trxs_mes_1 STORED AS PARQUET AS
SELECT cod_unico,
       cast(replace(left(cast(f_trx AS string), 6), '-', '') AS INT) AS periodo_trxs,
       count(*) AS num_trxs,
       sum(mnt_total_trxs) AS mnt_total_trxs
FROM proceso.mdo_wompi_vinc_primer_regi_trxs
GROUP BY 1,
       2;"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_trxs_mes_1;"""
helper.ejecutar_consulta(sql_compute)


##################################################

Obteniendo transacciones wompi de los comercios

-----------------------------------------------------------------------------------------------
  i    tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 5/5      DROP ...mdo_wompi_vinc_primer_regi_trxs_mes_1   finalizado   07:51:14 PM     00:00.5 
-----------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
  i    tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 6/6    CREATE ...mdo_wompi_vinc_primer_regi_trxs_mes_1   finalizado   07:51:15 PM     00:03.5 
-----------------------------------

## Construcción históricos

Se calcula la métrica uso para tres escenarios

- Todos los clientes
- Clientes nuevos
- Clientes viejos

Se dice que un cliente [todos, nuevos o viejos] tiene uso cuando realiza al menos una (1) transaccion en el año corriente. 

Importante tener en cuenta para clientes nuevos. Un cliente que vinculó wompy en el año 2024 [nuevo en 2024] y realizó un transacción en el mismo año suma a la métrica, en cambio si ese mismo cliente realiza una transacción en el año 2025 y siguiente no sumará a la métrica.

In [7]:
if actualizar == False:
    # Tabla que almacenará el número de vinculaciones por mes
    sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist  (
                    periodo DOUBLE,
                    num_vinc BIGINT,
                    tipo_cliente STRING
                    )
    STORED AS PARQUET
    TBLPROPERTIES ('transactional' = 'false');
    """
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist;"""
    helper.ejecutar_consulta(sql_compute)

-----------------------------------------------------------------------------------------------
  i    tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 8/8      DROP ...i_vinc_primer_regi_vinculaciones_hist   finalizado   07:51:32 PM     00:00.5 
-----------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
  i    tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 9/9    CREATE ...i_vinc_primer_regi_vinculaciones_hist   finalizado   07:51:32 PM     00:00.5 
-----------------------------------------------------------------------------------------------
----------------------------------------

In [8]:
if actualizar == False:
    # Tabla que almacenará comercios con trxs por periodo y tipo cliente
    sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist  (
                    codigo_unico VARCHAR,
                    periodo DOUBLE,
                    num_trxs BIGINT,
                    mnt_total_trxs DECIMAL(38,2),
                    tipo_cliente STRING
                    )
    STORED AS PARQUET
    TBLPROPERTIES ('transactional' = 'false');
    """
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist;"""
    helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 11/11      DROP ...imer_regi_vinculaciones_con_trxs_hist   finalizado   07:51:49 PM     00:00.5 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 12/12    CREATE ...imer_regi_vinculaciones_con_trxs_hist   finalizado   07:51:49 PM     00:00.5 
-------------------------------------------------------------------------------------------------
--------------------

In [9]:
# Construcción histórico trxs
fecha_inicial_ts = pd.to_datetime(fecha_inicial_metricas)
fecha_final_ts = pd.to_datetime(fecha_final_metricas)
fechas = pd.date_range(start=fecha_inicial_metricas, end=fecha_final_metricas, freq='M')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config['periodo'] = df_config['year'] * 100 + df_config['month']
df_config['periodo_base_year'] = df_config['year'] * 100 + 1
df_config['periodo_sgte_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1))
df_config['year_sgte_mes'] = df_config['periodo_sgte_mes'].dt.year
df_config['periodo_sgte_mes'] = df_config['periodo_sgte_mes'].apply(lambda x: x.year * 100 + x.month)
df_config['periodo_viejos'] = df_config['fechas'].apply(lambda x: str(x.year - 1) + '12')
# df_config['ult_6_meses_fin'] = df_config['fechas'].apply(lambda x: x - relativedelta(months=5))
# df_config['ult_6_meses_inicio'] = df_config['ult_6_meses_fin'].apply(lambda x: x.replace(day=1))
# df_config['year_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.year
# df_config['month_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.month
# df_config['periodo_ult_6_meses'] = df_config['year_ult_6_meses'] * 100 + df_config['month_ult_6_meses']
df_config

,fechas,year,month,day,periodo,periodo_base_year,periodo_sgte_mes,year_sgte_mes,periodo_viejos
0,2022-01-31,2022,1,31,202201,202201,202202,2022,202112
1,2022-02-28,2022,2,28,202202,202201,202203,2022,202112
2,2022-03-31,2022,3,31,202203,202201,202204,2022,202112
3,2022-04-30,2022,4,30,202204,202201,202205,2022,202112
4,2022-05-31,2022,5,31,202205,202201,202206,2022,202112
5,2022-06-30,2022,6,30,202206,202201,202207,2022,202112
6,2022-07-31,2022,7,31,202207,202201,202208,2022,202112
7,2022-08-31,2022,8,31,202208,202201,202209,2022,202112
8,2022-09-30,2022,9,30,202209,202201,202210,2022,202112
9,2022-10-31,2022,10,31,202210,202201,202211,2022,202112


In [10]:
dict_ult_ing_wompi_vinc_primer_regi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_vinc_primer_regi_merch

2026-08-13 19:52:03 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-08-13 19:52:04 - [INFO] - Finalizo la busqueda, duracion: 00:01.3, resultado: {'year': 2026, 'month': 8, 'day': 13}


{'year': 2026, 'month': 8, 'day': 13}

In [11]:
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Obteniendo Métrica 5X')
    print('')
    print('Mes de Análisis: ', str(row.year), '-', str(row.month))
    print('')
    print('Viejos')
    print('')
    print('Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT id_comercio as codigo_unico,
              min(cast(left(cast(creado AS string), 6) AS int)) AS periodo
       FROM resultados_wompi.wompi_merchants
       WHERE YEAR = """ + str(dict_ult_ing_wompi_vinc_primer_regi_merch['year']) + """
       AND MONTH = """ + str(dict_ult_ing_wompi_vinc_primer_regi_merch['month']) + """
       AND DAY = """ + str(dict_ult_ing_wompi_vinc_primer_regi_merch['day']) + """
       AND modelo = 'Agregador'
       AND UPPER(activo) = 'A'
       AND UPPER(desembolsos_permitidos) = 'SI'
       AND UPPER(plan_dispersion) <> 'PLAN FREMIUM NEQUI NEGOCIOS'
       AND UPPER(plan_dispersion) <> 'PLAN NEQUI NEGOCIOS EMPRENDEDORES'
       GROUP BY 1
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM vinc
    WHERE periodo <= """ + str(row.periodo_viejos) + """;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando número de vinculados viejos en proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist')
    print('')
    sql = """
    INSERT INTO proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist
    SELECT periodo,
           count(*) AS num_vinc,
           'viejos' as tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'viejos' as tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist
    SELECT cast(a.codigo_unico as varchar) as codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_wompi_vinc_primer_regi_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')

    sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist;"""
    helper.ejecutar_consulta(sql_compute)


    print('Nuevos')
    print('')
    print('Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT id_comercio as codigo_unico,
              min(cast(left(cast(creado AS string), 6) AS int)) AS periodo
       FROM resultados_wompi.wompi_merchants
       WHERE YEAR = """ + str(dict_ult_ing_wompi_vinc_primer_regi_merch['year']) + """
       AND MONTH = """ + str(dict_ult_ing_wompi_vinc_primer_regi_merch['month']) + """
       AND DAY = """ + str(dict_ult_ing_wompi_vinc_primer_regi_merch['day']) + """
       AND modelo = 'Agregador'
       AND UPPER(activo) = 'A'
       AND UPPER(desembolsos_permitidos) = 'SI'
       AND UPPER(plan_dispersion) <> 'PLAN FREMIUM NEQUI NEGOCIOS'
       AND UPPER(plan_dispersion) <> 'PLAN NEQUI NEGOCIOS EMPRENDEDORES'
       GROUP BY 1
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM vinc
    WHERE periodo = """ + str(row.periodo) + """;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando número de vinculados nuevos en proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist')
    print('')
    sql = """
    INSERT INTO proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist
    SELECT periodo,
           count(*) AS num_vinc,
           'nuevos' as tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')

    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'nuevos' as tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist
    SELECT cast(a.codigo_unico as varchar) as codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_wompi_vinc_primer_regi_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')

    sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist;"""
    helper.ejecutar_consulta(sql_compute)

    print('Todos')
    print('')
    print('Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT id_comercio as codigo_unico,
              min(cast(left(cast(creado AS string), 6) AS int)) AS periodo
       FROM resultados_wompi.wompi_merchants
       WHERE YEAR = """ + str(dict_ult_ing_wompi_vinc_primer_regi_merch['year']) + """
       AND MONTH = """ + str(dict_ult_ing_wompi_vinc_primer_regi_merch['month']) + """
       AND DAY = """ + str(dict_ult_ing_wompi_vinc_primer_regi_merch['day']) + """
       AND modelo = 'Agregador'
       AND UPPER(activo) = 'A'
       AND UPPER(desembolsos_permitidos) = 'SI'
       AND UPPER(plan_dispersion) <> 'PLAN FREMIUM NEQUI NEGOCIOS'
       AND UPPER(plan_dispersion) <> 'PLAN NEQUI NEGOCIOS EMPRENDEDORES'
       GROUP BY 1
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM vinc
    WHERE periodo <= """ + str(row.periodo) + """;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando número de vinculados todos en proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist')
    print('')
    sql = """
    INSERT INTO proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist
    SELECT periodo,
           count(*) AS num_vinc,
           'todos' as tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'todos' as tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    print(sql)
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist
    SELECT cast(a.codigo_unico as varchar) as codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_wompi_vinc_primer_regi_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')

    sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist;"""
    helper.ejecutar_consulta(sql_compute)

    

##################################################

Obteniendo Métrica 5X

Mes de Análisis:  2022 - 1

Viejos

Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 14/14      DROP ...i_vinc_primer_regi_vinculaciones_temp   finalizado   07:52:26 PM     00:00.6 
-------------------------------------------------------------------------------------------------

    CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT id_comercio as codigo_unico,
              min(cast(left(cast(creado AS string), 6) AS int)) AS periodo
       FROM resultados_wompi.wompi_merchants
       WHERE YEAR = 2026
       AND MO

# Evitar ciclo de vida zona proceso_vdm

In [13]:
# Tabla proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist'

sql_drop_1 = f"""DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_1)

sql_proceso = f"""CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_vinculaciones_hist STORED AS PARQUET AS
    SELECT periodo,
           num_vinc,
           tipo_cliente
    FROM proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_proceso)

sql_compute_1 = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_compute_1)

sql_drop_2 = f"""DROP TABLE IF EXISTS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_2)

sql_vdm = f"""CREATE TABLE proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist STORED AS PARQUET AS
    SELECT periodo,
           num_vinc,
           tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_vdm)

sql_compute_2 = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_compute_2)

sql_drop_3 = f"""DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_hist PURGE;"""

-----------------------------------------------------------------------------------------------------
     i       tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------------
 1551/1551      DROP ...i_vinc_primer_regi_vinculaciones_hist   finalizado   11:43:57 AM     00:00.6 
-----------------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------------
     i       tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------------
 1552/1552    CREATE ...i_vinc_primer_regi_vinculaciones_hist   finalizado   11:43:57 AM     00:02.2 
----------------------------------------------------------------------------------

In [14]:
# Tabla proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist

sql_drop_1 = f"""DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_1)

sql_proceso = f"""CREATE TABLE proceso.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist STORED AS PARQUET AS
    SELECT codigo_unico,
           periodo,
           num_trxs,
           mnt_total_trxs,
           tipo_cliente
    FROM proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_proceso)

sql_compute_1 = """COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_compute_1)

sql_drop_2 = f"""DROP TABLE IF EXISTS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_2)

sql_vdm = f"""CREATE TABLE proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist STORED AS PARQUET AS
    SELECT codigo_unico,
           periodo,
           num_trxs,
           mnt_total_trxs,
           tipo_cliente
    FROM proceso.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_vdm)

sql_compute_2 = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_compute_2)

sql_drop_3 = f"""DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_regi_vinculaciones_con_trxs_hist PURGE;"""
helper.ejecutar_consulta(sql_drop_3)

-----------------------------------------------------------------------------------------------------
     i       tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------------
 1557/1557      DROP ...imer_regi_vinculaciones_con_trxs_hist   finalizado   11:44:04 AM     00:00.6 
-----------------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------------
     i       tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------------
 1558/1558    CREATE ...imer_regi_vinculaciones_con_trxs_hist   finalizado   11:44:05 AM     00:02.8 
----------------------------------------------------------------------------------